In [21]:
import matplotlib.pyplot as plt
import numpy as np
import os 
from pathlib import Path
import torch 

import m2aia as m2
from IMSAutoEncoderWrapper import IMSPyTorchDataset, IMSAutoEncoder
from IMSAutoEncoderWrapper.utils.Binners import LinearBinning, NotEmptyInverseBinner, TopPeaksInverseBinner

# seting global dir
cwd=Path.cwd()
if cwd.name == "tutorials":
    # os.chdir(cwd.parent.parent) 
    os.chdir(cwd.parent.parent) 
os.getcwd()

'/home/maxi7524/repositories/pym2aia_test'

## Introduction

In this tutorial we are going to:
1. Reconstruction 
   - load preexisting model and compressed latent space 
   - reconstruct it into `imzML` format file 
2. Visualization
   - 

## 1. Reconstruction 

To reconstruct file we first load our model. All needed attributes are already loaded

In [22]:
# instantiation model
PATH_DATA = Path('data') / 'bladder_data'
model_path = PATH_DATA / 'compressed_bladder_data' / 'model'
model = IMSAutoEncoder(path=model_path)

# load pretrained model
model.load()

Model will be loaded on cuda
[Manager] Loading and reconstructing model from data/bladder_data/compressed_bladder_data/model
[OK] Component reconstructed: LinearBinning
[OK] Component reconstructed: TopPeaksInverseBinner
[OK] Architecture reconstructed: ContrastiveAutoencoderSkrajny
[FAILED] Criterion ContrastiveCriterion not in registry.
[OK] Weights loaded: model_weights.pt
[OK] History loaded (10 epochs)
--- Model Load Complete ---



To reconstruct file we run `reconstruct_from_file` on loaded model

In [4]:
path_compressed_npz = PATH_DATA / 'compressed_bladder_data' / 'mouse_bladder_compressed.npz'
path_name_new_imzML_file = PATH_DATA / 'compressed_bladder_data' / 'mouse_bladder_reconstruct'

model.reconstruct_from_file(
    compressed_img_path=path_compressed_npz, 
    output_imzml=path_name_new_imzML_file)

[Model] Reconstructing image from data/bladder_data/compressed_bladder_data/mouse_bladder_compressed.npz...
[Model] Reconstructing 34840 spectra using batch processing...
[  5%] 1792/34840 pixels processed | ETA: 3.0 min
[ 10%] 3584/34840 pixels processed | ETA: 2.7 min
[ 15%] 5376/34840 pixels processed | ETA: 2.4 min
[ 21%] 7168/34840 pixels processed | ETA: 2.3 min
[ 26%] 8960/34840 pixels processed | ETA: 2.1 min
[ 31%] 10752/34840 pixels processed | ETA: 1.9 min
[ 36%] 12544/34840 pixels processed | ETA: 1.8 min
[ 41%] 14336/34840 pixels processed | ETA: 1.7 min
[ 46%] 16128/34840 pixels processed | ETA: 1.5 min
[ 51%] 17920/34840 pixels processed | ETA: 1.4 min
[ 57%] 19712/34840 pixels processed | ETA: 1.2 min
[ 62%] 21504/34840 pixels processed | ETA: 1.1 min
[ 67%] 23296/34840 pixels processed | ETA: 0.9 min
[ 72%] 25088/34840 pixels processed | ETA: 0.8 min
[ 77%] 26880/34840 pixels processed | ETA: 0.6 min
[ 82%] 28672/34840 pixels processed | ETA: 0.5 min
[ 87%] 30464/34840

To visualize both images we use `napari` library

In [16]:
# original data
PATH_real_data = PATH_DATA / 'bladder_data' / 'mouse_urinary_bladder.imzML'
path_name_new_imzML_file = PATH_DATA / 'compressed_bladder_data' / 'mouse_bladder_reconstruct'

print(PATH_real_data.exists())
I_original = m2.ImzMLReader(str(PATH_real_data), normalization=m2.m2NormalizationTIC )


# autoencoded data
PATH_autoencoded_data = Path(str(path_name_new_imzML_file) + '.imzML')
I_autoencoded =  m2.ImzMLReader(str(PATH_autoencoded_data))



True
!38.495! WARNING: No pixel size found, set x and y spacing to 50 microns!
!41.649! WARNING: Processed profile spectrum is not fully supported! Check the ImzML file.
[42.478] [imzML]: data/bladder_data/bladder_data/mouse_urinary_bladder.imzML
	[pixel size (mm)]: 0.050000x0.050000x0.010000
	[image dimension]: 260x134x1
	[num spectra]: 34840
	[spec. type]: ProcessedProfile
	[mass range]: 400.259620 to 999.795899 with #1500 measurements
	[normalization]: TIC
!42.480! WARNING: No pixel size found, set x and y spacing to 50 microns!
[46.142] [imzML]: data/bladder_data/compressed_bladder_data/mouse_bladder_reconstruct.imzML
	[pixel size (mm)]: 0.050000x0.050000x0.010000
	[image dimension]: 260x134x1
	[num spectra]: 34840
	[spec. type]: ProcessedCentroid
	[mass range]: 404.960003 to 994.120001 with #284 measurements


## 2. Visualization

To visualize our results we can use either our helpers or use napari


#TODO 
Możemy zrobić z tego osobny notebook który pokazuje jak plotować co i po co żeby widzieć różnice

### Our visualization

In [ ]:
from IMSAutoEncoderWrapper.utils.VisualizationInteractive import M2AIAMultiExplorer
visualizer = M2AIAMultiExplorer([I_original, I_autoencoded])

In [18]:
# REMARK: first pixel is missing 
visualizer.plot()

ValueError: ('Center is out of x-axis range!', np.float64(400.2596199440525), 0.1, <class 'numpy.float32'>, False)

### Napari

In [19]:
!uv pip install napari

Using Python 3.12.13 environment at: /home/maxi7524/micromamba/envs/ims_env
Checked 1 package in 228ms


In [20]:
import napari


# 1. Fix graphical rendering issues for WSL/Remote environments
os.environ['LIBGL_ALWAYS_SOFTWARE'] = '1'

def get_reshaped_ion_image(reader, mz, tolerance):
    """
    Helper to fetch an ion image from m2aia and reshape it to (z, y, x)
    """
    nx, ny, nz = reader.GetShape()
    # GetImage returns an m2aia object, cast to numpy and reshape
    img_data = np.array(reader.GetImage(mz, tolerance))
    return img_data.reshape((nz, ny, nx))

# 2. Load two separate images for comparison
# Assuming I_loaded is your first image, let's define a second one
# reader1 = I_loaded
# reader2 = m2.ImzMLReader("path_to_second_file.imzML")
# reader2.Load()

reader1 = I_original
# For demonstration, I will use the same reader as reader2 if you only have one loaded
reader2 = I_autoencoded

# 3. Initial parameters
target_mz = 800.5
tolerance = 0.1

# 4. Initialize Napari Viewer
viewer = napari.Viewer()

# 5. Load initial ion images for both datasets
ion1 = get_reshaped_ion_image(reader1, target_mz, tolerance)
ion2 = get_reshaped_ion_image(reader2, target_mz, tolerance)

layer1 = viewer.add_image(ion1, name='Dataset A', colormap='magma')
layer2 = viewer.add_image(ion2, name='Dataset B', colormap='magma')

# 6. Enable Grid View for side-by-side comparison
# This puts the images next to each other instead of overlapping
viewer.grid.enabled = True
viewer.grid.shape = (1, 2) # 1 row, 2 columns

# 7. Interactive M/Z Update Function
def update_mz(mz):
    """
    Call this function in a new notebook cell to change the mass on both images
    Example: update_mz(760.4)
    """
    print(f"Updating images to {mz} m/z...")
    layer1.data = get_reshaped_ion_image(reader1, mz, tolerance)
    layer2.data = get_reshaped_ion_image(reader2, mz, tolerance)
    layer1.name = f'A: {mz} m/z'
    layer2.name = f'B: {mz} m/z'

# 8. Add Masks if available
try:
    nx1, ny1, nz1 = reader1.GetSize()
    mask1 = np.array(reader1.GetMaskImage()).reshape((nz1, ny1, nx1))
    viewer.add_labels(mask1.astype(int), name='Mask A', opacity=0.2)
    
    nx2, ny2, nz2 = reader2.GetSize()
    mask2 = np.array(reader2.GetMaskImage()).reshape((nz2, ny2, nx2))
    viewer.add_labels(mask2.astype(int), name='Mask B', opacity=0.2)
except Exception as e:
    print(f"No masks loaded: {e}")

# To browse a new mass, simply run this in the next cell:
# update_mz(742.5)

No masks loaded: 'ImzMLReader' object has no attribute 'GetSize'
